# Per-UID LSTM for IEEE-CIS Fraud — PyTorch version

This notebook mirrors `Time_Series_LSTM_per_UID.ipynb` (Keras) but is written in
PyTorch, following the structural pattern from the Kaggle reference:

> [arunmohan003 — *Sentiment analysis using LSTM - PyTorch*](https://www.kaggle.com/code/arunmohan003/sentiment-analysis-using-lstm-pytorch)

Same upstream logic as the Keras notebook:

- Load preprocessed checkpoints (`X_train_copy4.pkl`, `X_test_copy4.pkl`, `y_train.pkl`).
- Bridge train+test before windowing so test rows can use train history.
- Add three time-gap features per UID.
- Standardize (fit on train rows only).
- Build per-UID sliding windows of length `WINDOW`.
- Train under **strict expanding-window time validation** with `MIN_TRAIN_MONTHS = 3`.
- Save OOF and test predictions for ensembling with your XGBoost OOF.

## What's adapted from the sentiment-analysis reference

| arunmohan003's notebook | This notebook |
|--|--|
| Word indices `(batch, seq_len)` → `nn.Embedding` → `(batch, seq_len, embed_dim)` | Numeric features `(batch, seq_len, n_features)` directly into LSTM (no embedding) |
| `vocab_size`, `embedding_dim` hyperparameters | `n_features` only — no vocab |
| `nn.LSTM(input_size=embedding_dim, ...)` | `nn.LSTM(input_size=n_features, ...)` |
| `model.init_hidden(batch_size)` per iteration | Same pattern, kept for fidelity |
| `nn.BCELoss` after sigmoid | Same |
| Manual training loop with `optimizer.zero_grad()`, `loss.backward()`, `clip_grad_norm_`, `optimizer.step()` | Same |
| Best model saved by validation loss | Best model saved by **validation AUC** (better metric for fraud) |

The reason there's no embedding layer: in sentiment analysis each word is a discrete
token that has to be turned into a continuous vector. Your fraud features are already
continuous after `StandardScaler` (and previously-categorical fields like `card1` were
already integer-encoded by the upstream pipeline), so the LSTM can ingest them directly.


## MPS / Apple Silicon GPU notes

PyTorch supports Apple Silicon GPU via the **MPS** (Metal Performance Shaders) backend.
The config cell below picks the best available device automatically: `mps` if you're on
M-series, else `cuda`, else `cpu`.

A few specifics for MPS:

- Some ops fall back to CPU silently. For LSTM this works but you may see warnings
  about unsupported dtypes — mostly harmless.
- `torch.compile(...)` doesn't help much on MPS yet (Metal backend is limited),
  so this notebook doesn't use it.
- Mixed-precision (`autocast`) is supported but not used here — fp32 is more stable
  for LSTM on MPS, and the speed difference is small.
- `num_workers > 0` in `DataLoader` can deadlock on macOS in Jupyter. We use
  `num_workers=0` and rely on the unified-memory architecture for fast host-device
  transfer.


In [1]:
import sys, os
print(sys.executable)
print(os.environ.get("CONDA_DEFAULT_ENV"))

/Users/hovietbach/miniforge3/envs/Financial_Fraud_Detection_Thesis/bin/python
Financial_Fraud_Detection_Thesis


In [2]:
# 0. Imports and config — PyTorch + MPS-aware
import os, gc, math, time, datetime, warnings, copy
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# ----- Configuration -----
DATA_DIR = '/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/pkl_exported_files'

WINDOW              = 20      # was 5  (change F)
MIN_TRAIN_MONTHS    = 3
BATCH               = 1024
EPOCHS              = 30      # was 12 (change I)
LR                  = 1e-3
WEIGHT_DECAY        = 2e-4
GRAD_CLIP           = 1.0     # was 0.5 (change D)
EARLY_STOP_PATIENCE = 6
SEED                = 42

HIDDEN_DIM   = 128
NUM_LAYERS   = 2
DROPOUT      = 0.3
N_SEEDS      = 3              # for seed ensembling (change H)
USE_POS_WEIGHT = False        # plain BCE for AUC (change C)
USE_STATIC_TOWER = False       # dual-tower (change B)

# ----- Reproducibility -----
torch.manual_seed(SEED); np.random.seed(SEED)

# ----- Device selection -----
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'PyTorch {torch.__version__}  device={device}')


PyTorch 2.10.0  device=mps


In [3]:
x = torch.randn(1024, 5, 250).to(device)
print(x.device)

mps:0


## 1. Load preprocessed checkpoints (pickle)


In [4]:
X_train = pd.read_pickle(os.path.join(DATA_DIR, 'X_train_copy4.pkl'))
X_test  = pd.read_pickle(os.path.join(DATA_DIR, 'X_test_copy4.pkl'))
y_train = pd.read_pickle(os.path.join(DATA_DIR, 'y_train.pkl')).astype('int8')

print('train rows :', len(X_train))
print('test  rows :', len(X_test))
print('train cols :', X_train.shape[1])
print('positive rate:', y_train.mean().round(4))

train rows : 590540
test  rows : 506691
train cols : 266
positive rate: 0.035


In [5]:
import pyarrow
print(pyarrow.__version__)

24.0.0


## 5. Build per-UID windows on the combined frame

In [ ]:
data = np.load("/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/split_data.npz")

X_train_seq = data["X_train_seq"]
L_train = data["L_train"]
X_test_seq = data["X_test_seq"]
L_test = data["L_test"]
train_order = data["train_order"]
test_order = data["test_order"]
train_pos = data["train_pos"]
test_pos = data["test_pos"]
y_aligned = data["y_aligned"]
dt_m_aligned = data["dt_m_aligned"]

## The above is currently the left-padding, therefore, we need to add a method to convert that left-padding sequence to right-padidng, or we need to build the sequence again following the righ-padding style. Otherwise, the below code will crash.

In [ ]:
def infer_padding_side(X, lengths, sample_size=20_000, name='X'):
    """Infer whether variable-length windows are left- or right-padded.

    Padding rows are expected to be exactly all-zero rows. Full-length rows are
    ignored because they contain no padding and therefore cannot identify side.
    """
    lengths = np.asarray(lengths, dtype=np.int64)
    N, T = X.shape[:2]
    if len(lengths) != N:
        raise ValueError(f'{name}: lengths has {len(lengths):,} rows but X has {N:,}')

    valid = np.flatnonzero(lengths < T)
    if valid.size == 0:
        return 'none'

    if valid.size > sample_size:
        pick = np.linspace(0, valid.size - 1, sample_size, dtype=np.int64)
        valid = valid[pick]

    left_ok = 0
    right_ok = 0
    checked = 0
    for i in valid:
        L = int(lengths[i])
        if L <= 0 or L > T:
            raise ValueError(f'{name}: invalid length {L} at row {i}')
        seq = X[i]
        left_pad_zero = not np.any(seq[:T - L])
        right_pad_zero = not np.any(seq[L:])
        left_ok += int(left_pad_zero)
        right_ok += int(right_pad_zero)
        checked += 1

    left_rate = left_ok / max(checked, 1)
    right_rate = right_ok / max(checked, 1)
    print(f'{name}: padding side check on {checked:,} rows -> left={left_rate:.3f}, right={right_rate:.3f}')

    if left_rate > 0.99 and right_rate < 0.01:
        return 'left'
    if right_rate > 0.99 and left_rate < 0.01:
        return 'right'
    if left_rate > 0.99 and right_rate > 0.99:
        return 'ambiguous'
    return 'mixed'


def left_to_right_padded_inplace(X, lengths, chunk_size=8192, name='X'):
    """Convert left-padded windows to right-padded windows while preserving order.

    Example with T=5 and length=3:
        [PAD, PAD, txn1, txn2, txn3] -> [txn1, txn2, txn3, PAD, PAD]

    The conversion is chunked to avoid allocating a second full copy of X.
    The returned array is right-padded and may be the same object as X.
    """
    lengths = np.asarray(lengths, dtype=np.int64)
    N, T, F = X.shape
    if len(lengths) != N:
        raise ValueError(f'{name}: lengths has {len(lengths):,} rows but X has {N:,}')
    if np.any(lengths < 1) or np.any(lengths > T):
        bad = np.flatnonzero((lengths < 1) | (lengths > T))[:5]
        raise ValueError(f'{name}: invalid sequence lengths at rows {bad.tolist()}')

    if not X.flags.writeable:
        X = X.copy()

    side = infer_padding_side(X, lengths, name=name)
    if side in ('right', 'none', 'ambiguous'):
        print(f'{name}: already compatible with right-padding; no conversion needed.')
        return X
    if side != 'left':
        raise ValueError(f'{name}: expected left-padded cache, detected {side!r}; refusing to convert automatically.')

    print(f'{name}: converting left-padding -> right-padding, shape={X.shape}, dtype={X.dtype}')
    for start in range(0, N, chunk_size):
        end = min(start + chunk_size, N)
        chunk = X[start:end]
        original = chunk.copy()
        chunk.fill(0)
        L = lengths[start:end]

        # For each destination timestep d, copy the corresponding real timestep
        # from its original left-padded location T - length + d.
        for d in range(T):
            rows = np.flatnonzero(L > d)
            if rows.size == 0:
                continue
            src_pos = T - L[rows] + d
            chunk[rows, d, :] = original[rows, src_pos, :]

    converted_side = infer_padding_side(X, lengths, name=f'{name} after conversion')
    if converted_side not in ('right', 'none', 'ambiguous'):
        raise RuntimeError(f'{name}: conversion failed; detected {converted_side!r}')
    return X


X_train_seq = left_to_right_padded_inplace(X_train_seq, L_train, name='X_train_seq')
X_test_seq = left_to_right_padded_inplace(X_test_seq, L_test, name='X_test_seq')

print('Right-padded cache ready:')
print('  X_train_seq:', X_train_seq.shape, X_train_seq.dtype)
print('  X_test_seq :', X_test_seq.shape, X_test_seq.dtype)
print('  L_train min/max:', int(L_train.min()), int(L_train.max()))
print('  L_test  min/max:', int(L_test.min()), int(L_test.max()))

## 6. PyTorch `Dataset` and `DataLoader`

A thin wrapper around the numpy arrays. We hand the Dataset whatever subset of indices
the current fold needs, so we don't have to copy big arrays.


In [82]:
def collate_sort_by_length(batch):
    if len(batch[0]) == 3:
        xb, lb, yb = zip(*batch)
        xb = torch.stack(xb)
        lb = torch.stack(lb)
        yb = torch.stack(yb)

        order = torch.argsort(lb, descending=True)
        return xb[order], lb[order], yb[order]

    xb, lb = zip(*batch)
    xb = torch.stack(xb)
    lb = torch.stack(lb)

    order = torch.argsort(lb, descending=True)
    return xb[order], lb[order]

In [83]:
class WindowDataset(Dataset):
    def __init__(self, X, lengths, y=None):
        self.X = torch.from_numpy(X)
        self.lengths = torch.from_numpy(lengths.astype('int64'))
        self.y = None if y is None else torch.from_numpy(y.astype('float32'))

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        if self.y is None:
            return self.X[i], self.lengths[i]
        return self.X[i], self.lengths[i], self.y[i]


def make_loader(X, lengths, y, batch_size, shuffle, sort_for_pack=False):
    return DataLoader(
        WindowDataset(X, lengths, y),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=2 if device.type == "cuda" else 0,
        pin_memory=(device.type == "cuda"),
        drop_last=False,
        collate_fn=collate_sort_by_length if sort_for_pack else None,
    )

## 7. PyTorch LSTM model

This is the structural mirror of arunmohan003's `SentimentRNN` — same `__init__` /
`forward` / `init_hidden` pattern — adapted for numeric input (no `nn.Embedding`).

### v5 update note

This version uses right-padded UID windows plus `pack_padded_sequence`.
Mean+max pooling is still kept after unpacking the LSTM output.
The static tower now reads the last real timestep by `lengths - 1`, not `x[:, -1, :]`.


In [84]:
class FraudLSTM(nn.Module):
    """LSTM with masked mean+max pooling and an optional static-row tower."""
    def __init__(self, n_features, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS,
                 drop_prob=DROPOUT, output_dim=1, use_static_tower=USE_STATIC_TOWER):
        super().__init__()
        self.use_static_tower = use_static_tower

        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=drop_prob if num_layers > 1 else 0.0,
        )

        seq_out_dim = 2 * hidden_dim  # mean + max

        if use_static_tower:
            self.static_mlp = nn.Sequential(
                nn.Linear(n_features, 256), nn.ReLU(), nn.Dropout(drop_prob),
                nn.Linear(256, 128),        nn.ReLU(), nn.Dropout(drop_prob),
                nn.Linear(128, 64),         nn.ReLU(),
            )
            combined_dim = seq_out_dim + 64
        else:
            combined_dim = seq_out_dim

        self.head = nn.Sequential(
            nn.Linear(combined_dim, 64), nn.ReLU(), nn.Dropout(drop_prob),
            nn.Linear(64, output_dim),
        )

    def forward(self, x, lengths, enforce_sorted=False):
        # UPDATED v5: packed LSTM over right-padded sequences; pooling retained
        # x: (B, T, F) right-padded; lengths: (B,) real-step counts
        B, T, _ = x.shape
        lengths_dev = lengths.to(x.device)

        lengths_cpu = lengths if lengths.device.type == "cpu" else lengths.detach().cpu()

        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths_cpu, batch_first=True, enforce_sorted=enforce_sorted
        )

        packed_out, _ = self.lstm(packed)

        # Do not force total_length=T; use max length in this batch.
        lstm_out, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True
        )

        T_eff = lstm_out.size(1)

        # Build a mask for the real left-aligned positions after right padding.
        idx = torch.arange(T_eff, device=x.device).unsqueeze(0)
        mask = idx < lengths_dev.unsqueeze(1)                              # (B, T)
        mask_f = mask.unsqueeze(-1).float()

        # masked mean
        sum_  = (lstm_out * mask_f).sum(dim=1)
        cnt   = mask_f.sum(dim=1).clamp(min=1.0)
        mean_pool = sum_ / cnt

        # masked max (pads -> very negative)
        neg_inf  = torch.finfo(lstm_out.dtype).min
        max_pool = lstm_out.masked_fill(~mask.unsqueeze(-1), neg_inf).max(dim=1).values

        seq_vec = torch.cat([mean_pool, max_pool], dim=1)

        if self.use_static_tower:
            row_idx = torch.arange(B, device=x.device)
            last_idx = (lengths_dev - 1).clamp(min=0)
            current = x[row_idx, last_idx, :]                      # last real step
            stat_vec = self.static_mlp(current)
            feat = torch.cat([seq_vec, stat_vec], dim=1)
        else:
            feat = seq_vec

        return self.head(feat).squeeze(-1)

In [85]:
# smoke test
N_FEATURES = X_train_seq.shape[2]
m = FraudLSTM(N_FEATURES).to(device)
xb = torch.randn(8, WINDOW, N_FEATURES, device=device)
lb = torch.randint(1, WINDOW+1, (8,), device='cpu')
print('forward smoke test out shape:', m(xb, lb).shape)
del m, xb, lb

forward smoke test out shape: torch.Size([8])


In [86]:
import torch

T = 6
lengths = torch.tensor([2, 4, 6])

idx = torch.arange(T).unsqueeze(0)

# UPDATED v5: right-padded sequences keep real steps at positions 0:length.
mask = idx < lengths.unsqueeze(1)
mask_f = mask.unsqueeze(-1).float()

print(idx)
print(lengths.unsqueeze(1))
print(mask)
print(mask_f)

tensor([[0, 1, 2, 3, 4, 5]])
tensor([[2],
        [4],
        [6]])
tensor([[ True,  True, False, False, False, False],
        [ True,  True,  True,  True, False, False],
        [ True,  True,  True,  True,  True,  True]])
tensor([[[1.],
         [1.],
         [0.],
         [0.],
         [0.],
         [0.]],

        [[1.],
         [1.],
         [1.],
         [1.],
         [0.],
         [0.]],

        [[1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.]]])


In [87]:
cnt   = mask_f.sum(dim=1).clamp(min=1.0)
print(cnt)

tensor([[2.],
        [4.],
        [6.]])


## 8. Expanding-window CV + training loop

Same fold scheme as the Keras notebook. Training loop mirrors arunmohan003's pattern:

1. Per epoch, iterate batches and reset hidden state for each batch (windows are
   independent).
2. `optimizer.zero_grad()` → `forward` → `loss.backward()` → `clip_grad_norm_` →
   `optimizer.step()`.
3. Validation pass with `model.eval()` and `torch.no_grad()`.
4. Early-stop on validation AUC, restore best weights.


In [88]:
device = torch.device('cuda' if torch.cuda.is_available()
                       else 'mps' if torch.backends.mps.is_available()
                       else 'cpu')
print('device:', device)


def expanding_month_folds(months_array, min_train_months=MIN_TRAIN_MONTHS):
    months = sorted(np.unique(months_array).tolist())
    for vm in months[min_train_months:]:
        tm = [m for m in months if m < vm]
        ti = np.flatnonzero(np.isin(months_array, tm))
        vi = np.flatnonzero(months_array == vm)
        yield (vm, tm, ti, vi)

device: mps


In [89]:
device = torch.device("cpu")
print('device:', device)

device: cpu


In [90]:
def train_one_fold(X_tr, L_tr, y_tr, X_va, L_va, y_va, n_features,
                   epochs, batch, lr, weight_decay, device,
                   early_stop_patience, grad_clip, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = FraudLSTM(n_features).to(device)

    if USE_POS_WEIGHT:
        pos = float((y_tr == 1).sum()); neg = float(len(y_tr) - pos)
        pw = torch.tensor([np.sqrt(neg / max(pos, 1.0))], device=device, dtype=torch.float32)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw)
    else:
        loss_fn = nn.BCEWithLogitsLoss()                            # plain BCE → better AUC

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    train_loader = make_loader(X_tr, L_tr, y_tr, batch_size=batch, shuffle=True, sort_for_pack=True)
    val_loader   = make_loader(X_va, L_va, y_va, batch_size=batch, shuffle=False, sort_for_pack=False)

    steps = max(1, math.ceil(len(X_tr) / batch))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=epochs, steps_per_epoch=steps,
        pct_start=0.1, anneal_strategy='cos',
    )

    best_auc, best_state, best_val_preds, bad = -1.0, None, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        t0 = time.time(); running, n_seen = 0.0, 0
        for xb, lb, yb in train_loader:
            xb = xb.to(device=device, dtype=torch.float32, non_blocking=True)
            yb = yb.to(device=device, dtype=torch.float32, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb, lb, enforce_sorted=True)
            loss = loss_fn(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step(); scheduler.step()
            running += loss.item() * xb.size(0); n_seen += xb.size(0)
        train_loss = running / max(n_seen, 1)

        model.eval(); preds = []
        with torch.no_grad():
            for xb, lb, _ in val_loader:
                xb = xb.to(device=device, dtype=torch.float32, non_blocking=True)
                preds.append(torch.sigmoid(model(xb, lb)).cpu().numpy())
        val_preds = np.concatenate(preds)
        val_auc = roc_auc_score(y_va, val_preds)

        print(f'   ep {epoch:>2}/{epochs}  loss={train_loss:.4f}  val_auc={val_auc:.4f}  ({time.time()-t0:.1f}s)')
        if val_auc > best_auc:
            best_auc = val_auc; best_state = copy.deepcopy(model.state_dict())
            best_val_preds = val_preds; bad = 0
        else:
            bad += 1
            if bad >= early_stop_patience:
                print(f'   early stop at epoch {epoch}'); break

    if best_state is not None: model.load_state_dict(best_state)
    return best_val_preds, best_auc, model

In [91]:
# ===== run folds with seed ensembling =====
oof        = np.full(len(X_train_seq), np.nan, dtype=np.float32)
test_preds = np.zeros(len(X_test_seq), dtype=np.float32)
fold_aucs  = []

fold_specs = list(expanding_month_folds(dt_m_aligned, MIN_TRAIN_MONTHS))
print(f'{len(fold_specs)} folds; {N_SEEDS} seeds per fold')

for fold, (vm, tm, idxT, idxV) in enumerate(fold_specs):
    print(f'\n=== Fold {fold}: train {tm} → validate {vm} '
          f'(train={len(idxT):,}, valid={len(idxV):,}) ===')

    seed_val_preds, seed_test_preds = [], []
    for s in range(N_SEEDS):
        seed = SEED + s
        print(f'-- seed {seed} --')
        vp, va, model = train_one_fold(
            X_train_seq[idxT], L_train[idxT], y_aligned[idxT],
            X_train_seq[idxV], L_train[idxV], y_aligned[idxV],
            n_features=N_FEATURES, epochs=EPOCHS, batch=BATCH,
            lr=LR, weight_decay=WEIGHT_DECAY, device=device,
            early_stop_patience=EARLY_STOP_PATIENCE,
            grad_clip=GRAD_CLIP, seed=seed,
        )
        seed_val_preds.append(vp)

        # test preds from this seed's best model
        model.eval()
        test_loader = make_loader(X_test_seq, L_test, y=None, batch_size=BATCH, shuffle=False)
        tps = []
        with torch.no_grad():
            for xb, lb in test_loader:
                xb = xb.to(device=device, dtype=torch.float32, non_blocking=True)
                tps.append(torch.sigmoid(model(xb, lb)).cpu().numpy())
        seed_test_preds.append(np.concatenate(tps))

        del model
        if device.type == 'mps': torch.mps.empty_cache()
        gc.collect()

    fold_val   = np.mean(seed_val_preds, axis=0)
    fold_test  = np.mean(seed_test_preds, axis=0)
    fold_auc   = roc_auc_score(y_aligned[idxV], fold_val)
    print(f'   fold AUC (seed-avg) = {fold_auc:.4f}')
    fold_aucs.append((int(vm), float(fold_auc)))
    oof[idxV]  = fold_val
    test_preds += fold_test

if len(fold_specs):
    test_preds /= len(fold_specs)

validated = ~np.isnan(oof)
overall_auc = roc_auc_score(y_aligned[validated], oof[validated])
print(f'\n=== LSTM OOF AUC (validated months only) = {overall_auc:.4f} ===')
print(f'   per-fold: {fold_aucs}')

3 folds; 3 seeds per fold

=== Fold 0: train [12, 13, 14] → validate 15 (train=315,927, valid=101,632) ===
-- seed 42 --
   ep  1/30  loss=0.3995  val_auc=0.8120  (40.2s)
   ep  2/30  loss=0.1090  val_auc=0.8590  (37.2s)
   ep  3/30  loss=0.0894  val_auc=0.8731  (37.2s)
   ep  4/30  loss=0.0800  val_auc=0.8737  (39.9s)
   ep  5/30  loss=0.0731  val_auc=0.8638  (39.0s)
   ep  6/30  loss=0.0679  val_auc=0.8743  (36.1s)
   ep  7/30  loss=0.0647  val_auc=0.8795  (36.6s)
   ep  8/30  loss=0.0609  val_auc=0.8660  (40.9s)
   ep  9/30  loss=0.0574  val_auc=0.8554  (38.8s)
   ep 10/30  loss=0.0550  val_auc=0.8706  (38.8s)
   ep 11/30  loss=0.0525  val_auc=0.8569  (41.2s)
   ep 12/30  loss=0.0500  val_auc=0.8632  (42.0s)
   ep 13/30  loss=0.0480  val_auc=0.8641  (39.1s)
   early stop at epoch 13
-- seed 43 --
   ep  1/30  loss=0.3499  val_auc=0.8227  (39.3s)
   ep  2/30  loss=0.1057  val_auc=0.8586  (38.8s)
   ep  3/30  loss=0.0865  val_auc=0.8732  (39.1s)
   ep  4/30  loss=0.0775  val_auc=0.874